# VQA Múa Lân — Main Training Notebook

Huấn luyện **2 cấu hình tối ưu nhất** và so sánh trực tiếp:

| Run | Decoder | Norm | FFN | Ghi chú |
|-----|---------|------|-----|---------|
| **A1** | LSTM      | —         | —       | Baseline rời (encoder-decoder cổ điển) |
| **A3** | Transformer | RMSNorm | SwiGLU  | Modern transformer stack (LLaMA/Gemma/Qwen) |

Encoder cố định: SigLIP2-B/16 (layer −2, frozen) + PhoBERT-v2 (mean 4 last layers, frozen) + Cross-Attention Fusion.
Loss: `CrossEntropy(ignore_index=PAD, label_smoothing=0.1)`.

Notebook này import trực tiếp từ `src/` — mọi class đã định nghĩa ở đó, không lặp code.

## 1. Setup

In [ ]:
import os, sys, time, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ModelConfig, TrainConfig
from src.build  import (build_tokenizer_and_processor, resolve_special_ids,
                        build_loaders, build_model)
from src.training import Trainer, Evaluator
from src.metrics  import ExactMatch, BLEUScore, METEORScore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Tokenizer & DataLoaders (dùng chung cho 2 run)

In [ ]:
_probe_cfg = ModelConfig()
tokenizer, image_processor = build_tokenizer_and_processor(_probe_cfg)
_probe_cfg = resolve_special_ids(tokenizer, _probe_cfg)

BATCH_SIZE = 16
EPOCHS     = 8
LR         = 3e-4

shared_train_cfg = TrainConfig(
    batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
    image_root=PROJECT_ROOT, num_workers=0,
    eval_every=1, save_every=EPOCHS,
)
train_loader, val_loader, test_loader = build_loaders(
    _probe_cfg, shared_train_cfg, tokenizer, image_processor)
print(f'train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  test={len(test_loader.dataset)}')

## 3. Trainer-with-history

Bọc lại `Trainer` để lưu `train_loss`, `val_acc`, `val_bleu`, `val_meteor` cho từng epoch — phục vụ vẽ biểu đồ line.

In [ ]:
class TrainerWithHistory(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.history = {'train_loss': [], 'val': {}}
    def fit(self):
        cfg = self.train_cfg
        for epoch in range(cfg.epochs):
            t0 = time.time()
            tl = self.train_one_epoch(epoch)
            self.history['train_loss'].append(tl)
            log = f'epoch={epoch+1}/{cfg.epochs} train_loss={tl:.4f} time={time.time()-t0:.1f}s'
            if self.evaluator is not None and self.val_loader is not None:
                res = self.evaluator.evaluate(self.model, self.val_loader)
                for k, v in res.items():
                    self.history['val'].setdefault(k, []).append(v)
                log += '  ' + ' '.join(f'val_{k}={v:.4f}' for k, v in res.items())
            self._log(log)
        self._save_ckpt('final')
        return self.history

In [ ]:
def run_config(tag, decoder_type, norm_type, ffn_type):
    print(f'\n========== {tag} ==========')
    mcfg = ModelConfig(decoder_type=decoder_type, norm_type=norm_type, ffn_type=ffn_type)
    mcfg = resolve_special_ids(tokenizer, mcfg)
    tcfg = TrainConfig(
        batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
        image_root=PROJECT_ROOT, num_workers=0,
        eval_every=1, save_every=EPOCHS, run_name=tag,
    )
    model = build_model(mcfg)
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    evaluator = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=mcfg.bos_id, eos_id=mcfg.eos_id, pad_id=mcfg.pad_id,
        max_len=mcfg.max_answer_len, beam_size=1,
    )
    trainer = TrainerWithHistory(
        model=model, train_loader=train_loader, val_loader=val_loader,
        train_cfg=tcfg, model_cfg=mcfg, evaluator=evaluator,
    )
    return trainer.fit()

## 4. Train A1 — LSTM

In [ ]:
hist_A1 = run_config('A1_lstm', decoder_type='lstm', norm_type='layernorm', ffn_type='vanilla')

## 5. Train A3 — Transformer + RMSNorm + SwiGLU

In [ ]:
hist_A3 = run_config('A3_transformer_modern', decoder_type='transformer', norm_type='rmsnorm', ffn_type='swiglu')

## 6. So sánh — biểu đồ line

In [ ]:
def plot_compare(histories, keys, title_prefix=''):
    n = len(keys)
    fig, axes = plt.subplots(1, n, figsize=(5.2*n, 4))
    if n == 1: axes = [axes]
    for ax, key in zip(axes, keys):
        for tag, h in histories.items():
            if key == 'train_loss':
                ys = h['train_loss']
            else:
                ys = h['val'].get(key, [])
            xs = list(range(1, len(ys)+1))
            ax.plot(xs, ys, marker='o', label=tag)
        ax.set_title(f'{title_prefix}{key}')
        ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

histories = {'A1_lstm': hist_A1, 'A3_transformer_modern': hist_A3}
plot_compare(histories, ['train_loss'])
plot_compare(histories, ['exact_match', 'bleu', 'meteor'], title_prefix='val_')

## 7. Lưu lại history

In [ ]:
os.makedirs('logs', exist_ok=True)
with open('logs/main_history.json', 'w', encoding='utf-8') as f:
    json.dump(histories, f, ensure_ascii=False, indent=2)
print('saved -> logs/main_history.json')